# Rule: **build_shipping_demand**

**Description**

Build regional demand for international navigation based on outflow volume of ports.

In **PyPSA-Spain**, a code line is included to export the file `european_ports.geojson`.

**Outputs**

- resources/`shipping_demand_s_{clusters}.csv`

In [ ]:
######################################## Parameters

### Run
name = 'case_SectorCoupled'
prefix = ''

### Network
clusters = 5

### Spatial domain 'ES' or 'EU' (for maps domain and NUTS regions)
spatial_domain = 'ES'

In [ ]:
##### Import packages
import matplotlib.pyplot as plt
import os 
import sys
import geopandas as gpd
import cartopy.crs as ccrs


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `shipping_demand_s_{clusters}.csv`

Load the file and preview its content.

In [ ]:
file = f"shipping_demand_s_{clusters}.csv"

df = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df.head()

## `european_ports.geojson`

Load the file and preview its content.

In [ ]:
file = 'european_ports.geojson'
path = f"{params["rootpath"]}/resources/{prefix}/{name}/"
gdf_ports = gpd.read_file(f"{path}{file}")

gdf_ports.head()

In [ ]:
#################### Parameters


# Fixed marker area in points^2. Relative outflow is shown by color.
marker_size = 150
label_fontsize = 8
label_dx = 0.3
label_dy = -0.10


#################### Figure
fig_size = [8, 8]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})

map_extent = params[f'boundaries_onshore_{spatial_domain}']
ax.set_extent(map_extent, crs=ccrs.PlateCarree())


### Add map features
xp.map_add_features(ax, params['map_add_features'])


### Prepare ports with valid geometry and positive outflows
ports_plot = (
    gdf_ports.copy()
    .dropna(subset=['geometry', 'properties_outflows'])
    .loc[lambda df: df['properties_outflows'] > 0]
)

n_ports = len(ports_plot)

if not ports_plot.empty:
    # Use representative points in case geometries are not Point.
    ports_plot = ports_plot.assign(plot_geometry=ports_plot.geometry.representative_point())
    ports_plot['x'] = ports_plot['plot_geometry'].x
    ports_plot['y'] = ports_plot['plot_geometry'].y

    max_outflow = ports_plot['properties_outflows'].max()
    ports_plot['normalized_outflow'] = ports_plot['properties_outflows'] / max_outflow

    scatter = ax.scatter(
        ports_plot['x'],
        ports_plot['y'],
        s=marker_size,
        c=ports_plot['normalized_outflow'],
        cmap='viridis',
        vmin=0,
        vmax=1,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.8,
        transform=ccrs.PlateCarree(),
        zorder=6,
    )

    colorbar = plt.colorbar(scatter, ax=ax, shrink=0.5, pad=0.02)
    colorbar.set_label('Relative outflow')

    # Label ports with high relative outflow using the port name column.
    high_outflow_ports = ports_plot[ports_plot['normalized_outflow'] > 0.5]

    for _, row in high_outflow_ports.iterrows():
        label_text = str(row['properties_Name'])

        ax.text(
            row['x'] + label_dx,
            row['y'] + label_dy,
            label_text,
            transform=ccrs.PlateCarree(),
            fontsize=label_fontsize,
            ha='left',
            va='bottom',
            color='black',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'),
            zorder=7,
        )

ax.set_title(f"Total ports: {n_ports}", fontsize=12)
plt.tight_layout()